In [7]:
import os
import shutil

wav_folder = "/vol/corpora/Rhapsodie/wav16k_corrected"
trn_file = "trn/ref.trn"
output_folder = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap"

os.makedirs(output_folder, exist_ok=True)

with open(trn_file, "r", encoding="utf-8") as f:
    lines = f.readlines()




In [8]:
for line in lines:
    line = line.strip()
    
    # Example format: speaker1.wav bonjour je suis ici
    parts = line.split("(")
    
    speaker_id = parts[1][:-1]
    transcript = parts[0]
    wav_name = speaker_id.split("-")[1]+"-"+ speaker_id.split("-")[0]+".wav"
    
    speaker_folder = os.path.join(output_folder, speaker_id)
    os.makedirs(speaker_folder, exist_ok=True)
    
    # Copy wav
    shutil.copy(
        os.path.join(wav_folder, wav_name),
        os.path.join(speaker_folder, wav_name)
    )
    
    # Write txt file
    txt_path = os.path.join(speaker_folder, speaker_id + ".txt")
    with open(txt_path, "w", encoding="utf-8") as txt_file:
        txt_file.write(transcript.lower())

In [9]:
import os
import subprocess

corpus_dir = "/vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap"

for root, dirs, files in os.walk(corpus_dir):
    for file in files:
        if file.endswith(".wav"):
            filepath = os.path.join(root, file)
            filerename = os.path.join(root, file.split(".")[0].split("-")[1]+"-"+file.split(".")[0].split("-")[0]+".wav")
            tmp_path = filepath.replace(".wav", "_tmp.wav")

            print(f"Processing {filepath}")

            command = [
                "ffmpeg",
                "-y",
                "-i", filepath,
                "-ac", "1",
                "-ar", "16000",
                "-sample_fmt", "s16",
                tmp_path
            ]

            result = subprocess.run(command, stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)

            if result.returncode == 0:
                os.replace(tmp_path, filepath)
                os.rename(filepath, filerename)
                print("✓ Replaced")
            else:
                print("✗ Failed")
                if os.path.exists(tmp_path):
                    os.remove(tmp_path)

print("Done.")


Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0011-Rhap/Rhap-M0011.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0015-Rhap/Rhap-M0015.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0003-Rhap/Rhap-M0003.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/D2003-Rhap/Rhap-D2003.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0022-Rhap/Rhap-M0022.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/D2004-Rhap/Rhap-D2004.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0001-Rhap/Rhap-M0001.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M0006-Rhap/Rhap-M0006.wav
✓ Replaced
Processing /vol/experiments3/imbenamor/TAPAS-FRAIS/data/mfa_data_ref/rhap/M2002-Rhap/Rhap-M2002.wav
✓ Replaced
P

In [12]:
from praatio import textgrid


def extract_phones_from_textgrid(tg_path, remove_silence=True,t=""):
    """
    Extract phoneme sequence and timestamps from MFA TextGrid.

    Returns:
        phones: list of phoneme labels
        intervals: list of (start, end, phone)
    """
    
    tg = textgrid.openTextgrid(tg_path, includeEmptyIntervals=True)
    
    # List available tiers
   # print("Available tiers:", tg.tierNames)
    
    # Usually MFA phoneme tier is named "phones"
    phone_tier = tg.getTier(t)
    
    phones = []
    intervals = []
    
    for start, end, label in phone_tier.entries:
        
        label = label.strip()
        
        # Skip empty intervals
        if label == "":
            continue
        
        # Optionally remove silence
        if remove_silence and label in ["sil", "sp", "spn"]:
            continue
        
        phones.append(label)
        intervals.append((start, end, label))
    
    return phones, intervals
 

In [15]:
d={}
for w in os.listdir(ref_tg_path):
    d[w]=w.split("-")[0]+"-"+w.split("-")[1]+".TextGrid"

In [13]:
ref_mapping = {
    # Consonants
    "Z": "ʒ",
    "S": "ʃ",
    "R": "ʁ",
    "N": "ŋ",
    "J": "ɲ",
    "H": "ɥ",
    "g": "ɡ",
    "Z=": "ʒ",

    # Vowels
    "E": "ɛ",
    "O": "ɔ",
    "2": "ø",
    "9": "œ",
    "@": "ə",

    # Nasals (SAMPA)
    "a~": "ɑ̃",
    "o~": "ɔ̃",
    "e~": "ɛ̃",
    "9~": "ɛ̃",
    "E": "ɛ",
    "m=": "m",
    "n=": "n",
    "9~": "ɛ̃",
}
hyp_mapping = {

    # Multilingual vowel variants
    "ɪ": "i",
    "ʊ": "u",
    "ɨ": "i",
    "ɜ": "ə",
    "ʌ": "ɔ",
    "ɒ": "ɔ",

    # Rhotic variants
    "ɣ": "ʁ",
    "ɹ": "ʁ",
    "ɾ": "ʁ",

    # Lateral variant
    "ʎ": "l",

    # Foreign consonants
    "β": "b",
    "θ": "t",
    "c": "k",
    "ɑ": "a",
    "mʲ": "m",
    "ɟ": "ɡ",
}


def normalize_ref(phones,mapping):
    return [mapping.get(p, p) for p in phones]
invalid_tokens = {"_", "%", "0", "?"}


In [18]:
ref_tg_path

'/vol/corpora/Rhapsodie/TextGrids-fev2013/'

In [19]:
from pathlib import Path
from jiwer import wer
import os
from jiwer import process_words
total_S = 0
total_D = 0
total_I = 0
total_N = 0
tg_path = Path("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/output_mfa/")
full_paths = [f.resolve() for f in tg_path.rglob("*.TextGrid")]
ref_tg_path ="/vol/corpora/Rhapsodie/TextGrids-fev2013/"
all_ref=[]
all_hyp=[]
for hyp in full_paths:
    for ref in os.listdir(ref_tg_path):
        if str(hyp).split("/")[-1]==d[ref]:
            phones, intervals = extract_phones_from_textgrid(hyp,t="phones")
            phones_ref, intervals_ref = extract_phones_from_textgrid(ref_tg_path+ref,t="phone")
            phones_ref = [p for p in phones_ref if p not in invalid_tokens]
            phones_ref = normalize_ref(phones_ref,ref_mapping)
            phones = normalize_ref(phones,hyp_mapping)
            ref_str = " ".join(phones_ref)
            hyp_str = " ".join(phones)
            out = process_words(ref_str, hyp_str)
        
            total_S += out.substitutions
            total_D += out.deletions
            total_I += out.insertions
            total_N += len(phones_ref)
            all_ref.append(phones_ref)
            all_hyp.append(phones)

corpus_per = 100 * (total_S + total_D + total_I) / total_N

print(f"Corpus PER: {corpus_per:.2f}%")

ZeroDivisionError: division by zero

In [98]:
hyp_set = set(p for utt in all_hyp for p in utt)
print(hyp_set)

{'v', 'ə', 'ɲ', 'e', 's', 'k', 'f', 't', 'ʁ', 'w', 'a', 'ɥ', 'ɔ', 'l', 'n', 'ɛ̃', 'ʒ', 'p', 'z', 'ɔ̃', 'ɑ̃', 'ɡ', 'ʃ', 'i', 'ŋ', 'm', 'u', 'œ', 'd', 'ɛ', 'j', 'y', 'b', 'ø', 'o'}


In [99]:
unique_phones = set(p for utt in all_ref for p in utt)
print(unique_phones)


{'v', 'ə', 'ɲ', 'e', 's', 'ʁ', 'f', 't', 'k', 'w', 'a', 'ɥ', 'ɔ', 'l', 'n', 'ɛ̃', 'ʒ', 'z', 'p', 'ɔ̃', 'ɑ̃', 'ɡ', 'ʃ', 'i', 'ŋ', 'm', 'u', 'œ', 'd', 'sjo~', 'ɛ', 'fe~', 'j', 'y', 'b', 'ø', 'o'}


In [77]:
substitution_counter = Counter()

for phones_ref, phones_hyp in zip(all_ref, all_hyp):

    ref_str = " ".join(phones_ref)
    hyp_str = " ".join(phones_hyp)

    out = process_words(ref_str, hyp_str)

    # Get alignment info
    for chunk in out.alignments:
        for op in chunk:
            if op.type == "substitute":
                ref_token = ref_str.split()[op.ref_start_idx]
                hyp_token = hyp_str.split()[op.hyp_start_idx]
                substitution_counter[(ref_token, hyp_token)] += 1

In [78]:
for (ref_p, hyp_p), count in substitution_counter.items():
    if ref_p == "9~":
        print(f"9~ → {hyp_p} : {count}")

9~ → e : 6
9~ → ɛ̃ : 369
9~ → ʁ : 2
9~ → ɔ̃ : 1
9~ → c : 1
9~ → y : 7
9~ → b : 1
9~ → a : 6
9~ → ɛ : 7
9~ → z : 1
9~ → ɲ : 1
9~ → n : 4
9~ → ə : 1
9~ → s : 1


In [88]:
from collections import Counter

flat_hyp = [p for utt in all_hyp for p in utt]
freq_hyp = Counter(flat_hyp)

for p in {'ʎ', 'mʲ', 'ɑ', 'c', 'ɟ'}:
    print(p, freq_hyp[p])


c 504
ɑ 193
ʎ 288
mʲ 33
ɟ 31


In [92]:
for (ref_p, hyp_p), count in substitution_counter.items():
    if hyp_p == "c":
        print(f"{ref_p} → c : {count}")
    if hyp_p == "ʎ":
        print(f"{ref_p} → ʎ : {count}")
    if hyp_p == "ɑ":
        print(f"{ref_p} → ɑ : {count}")
    if hyp_p == "mʲ":
        print(f"{ref_p} → mʲ : {count}")
    if hyp_p == "ɟ":
        print(f"{ref_p} → ɟ : {count}")

a → c : 3
l → ʎ : 265
ɡ → ɟ : 29
k → c : 328
p → c : 4
a → ɑ : 178
ʁ → c : 4
m → mʲ : 30
ə → c : 12
ʁ → ʎ : 1
9~ → c : 1
œ → c : 13
ʒ → c : 2
ə → mʲ : 1
d → c : 9
s → c : 33
z → c : 5
w → c : 1
n → ɑ : 1
f → c : 2
ə → ʎ : 1
z → ʎ : 1
ɛ̃ → c : 2
t → c : 4
e → c : 18
l → c : 6
m → c : 4
m → ɑ : 1
i → c : 1
j → ɑ : 1
b → c : 2
s → ʎ : 1
ø → c : 2
e → ʎ : 1
ɑ̃ → c : 1
t → ʎ : 1
d → ɑ : 1
ʒ → ɑ : 2
n → c : 1
ʃ → c : 1
v → ɟ : 1
ʒ → ʎ : 1
ɛ̃ → ɑ : 1
m → ɟ : 1
p → ʎ : 1
œ → ʎ : 1
z → ɑ : 1


In [83]:
from collections import Counter

flat_ref = [p for utt in all_ref for p in utt]
freq = Counter(flat_ref)

for phone, count in freq.items():
    if phone not in hyp_set:
        print(phone, count)

sjo~ 1
fe~ 1
Z= 1


In [20]:
from rapidfuzz.distance import Levenshtein

def align_sequences(ref, hyp):
    alignment = []
    ops = Levenshtein.editops(ref, hyp)

    ref_idx = hyp_idx = 0
    op_idx = 0

    while ref_idx < len(ref) or hyp_idx < len(hyp):
        if op_idx < len(ops) and \
           ops[op_idx].src_pos == ref_idx and \
           ops[op_idx].dest_pos == hyp_idx:

            op = ops[op_idx]

            if op.tag == "replace":
                alignment.append((ref_idx, hyp_idx))
                ref_idx += 1
                hyp_idx += 1

            elif op.tag == "delete":
                alignment.append((ref_idx, None))
                ref_idx += 1

            elif op.tag == "insert":
                alignment.append((None, hyp_idx))
                hyp_idx += 1

            op_idx += 1

        else:
            alignment.append((ref_idx, hyp_idx))
            ref_idx += 1
            hyp_idx += 1

    return alignment


In [21]:
import numpy as np

def boundary_errors(ref_intervals, hyp_intervals, alignment):
    
    start_errors = []
    end_errors = []
    duration_errors = []

    for ref_idx, hyp_idx in alignment:
        if ref_idx is not None and hyp_idx is not None:

            r_start, r_end, _ = ref_intervals[ref_idx]
            h_start, h_end, _ = hyp_intervals[hyp_idx]

            start_errors.append(abs(r_start - h_start))
            end_errors.append(abs(r_end - h_end))
            duration_errors.append(abs((r_end - r_start) -
                                       (h_end - h_start)))

    return start_errors, end_errors, duration_errors


In [32]:
stem

'M0021-Rhap'

In [34]:
import os
import numpy as np
from pathlib import Path
from jiwer import process_words
from rapidfuzz.distance import Levenshtein

# ------------------------------------------------------------------
# Paths
# ------------------------------------------------------------------

tg_path = Path("/vol/experiments3/imbenamor/TAPAS-FRAIS/data/output_mfa/")
ref_tg_path = Path("/vol/corpora/Rhapsodie/TextGrids-fev2013/")

full_paths = list(tg_path.rglob("*.TextGrid"))
ref_files = list(ref_tg_path.glob("*.TextGrid"))

# ------------------------------------------------------------------
# Build reference dictionary (stem → file path)
# ------------------------------------------------------------------

ref_dict={str(f.stem)[:-4].split("-")[1]+"-"+str(f.stem)[:-4].split("-")[0]:str(f) for f in ref_files}

# ------------------------------------------------------------------
# Containers
# ------------------------------------------------------------------

total_S = total_D = total_I = total_N = 0

all_start_errors = []
all_end_errors = []
all_duration_errors = []

# ------------------------------------------------------------------
# Alignment helper
# ------------------------------------------------------------------

def align_sequences(ref, hyp):
    alignment = []
    ops = Levenshtein.editops(ref, hyp)

    ref_idx = hyp_idx = 0
    op_idx = 0

    while ref_idx < len(ref) or hyp_idx < len(hyp):
        if op_idx < len(ops) and \
           ops[op_idx].src_pos == ref_idx and \
           ops[op_idx].dest_pos == hyp_idx:

            op = ops[op_idx]

            if op.tag == "replace":
                alignment.append((ref_idx, hyp_idx))
                ref_idx += 1
                hyp_idx += 1

            elif op.tag == "delete":
                alignment.append((ref_idx, None))
                ref_idx += 1

            elif op.tag == "insert":
                alignment.append((None, hyp_idx))
                hyp_idx += 1

            op_idx += 1
        else:
            alignment.append((ref_idx, hyp_idx))
            ref_idx += 1
            hyp_idx += 1

    return alignment


# ------------------------------------------------------------------
# Boundary error computation
# ------------------------------------------------------------------

def boundary_errors(ref_intervals, hyp_intervals, alignment):
    start_errors = []
    end_errors = []
    duration_errors = []

    for ref_idx, hyp_idx in alignment:
        if ref_idx is not None and hyp_idx is not None:
            if phones_ref[ref_idx] == phones_hyp[hyp_idx]:

                r_start, r_end, _ = ref_intervals[ref_idx]
                h_start, h_end, _ = hyp_intervals[hyp_idx]
    
                start_errors.append(abs(r_start - h_start))
                end_errors.append(abs(r_end - h_end))
                duration_errors.append(
                    abs((r_end - r_start) - (h_end - h_start))
                )

    return start_errors, end_errors, duration_errors


# ------------------------------------------------------------------
# Main loop
# ------------------------------------------------------------------

for hyp_path in full_paths:

    stem = hyp_path.stem
    if stem not in ref_dict.keys():
        continue

    ref_path = ref_dict[stem]
    #print(ref_path)
    # --- Extract phones and timestamps ---
    phones_hyp, hyp_intervals = extract_phones_from_textgrid(
        hyp_path, t="phones"
    )
    phones_ref, ref_intervals = extract_phones_from_textgrid(
        ref_path, t="phone"
    )

    # --- Remove invalid tokens (ref only) ---
    filtered_phones_ref = []
    filtered_intervals_ref = []

    for (start, end, label) in ref_intervals:
        if label not in invalid_tokens:
            filtered_phones_ref.append(label)
            filtered_intervals_ref.append((start, end, label))

    phones_ref = normalize_ref(filtered_phones_ref, ref_mapping)
    ref_intervals = filtered_intervals_ref


    # --- Normalize inventories ---
    phones_ref = normalize_ref(phones_ref, ref_mapping)
    phones_hyp = normalize_ref(phones_hyp, hyp_mapping)
    ref_offset = ref_intervals[0][0]
    hyp_offset = hyp_intervals[0][0]
    
    ref_intervals = [
        (start - ref_offset, end - ref_offset, label)
        for start, end, label in ref_intervals
    ]
    
    hyp_intervals = [
        (start - hyp_offset, end - hyp_offset, label)
        for start, end, label in hyp_intervals
    ]

    # --- PER computation ---
    ref_str = " ".join(phones_ref)
    hyp_str = " ".join(phones_hyp)

    out = process_words(ref_str, hyp_str)

    total_S += out.substitutions
    total_D += out.deletions
    total_I += out.insertions
    total_N += len(phones_ref)

    # --- Sequence alignment ---
    alignment = align_sequences(phones_ref, phones_hyp)

    # --- Boundary errors ---
    s_err, e_err, d_err = boundary_errors(
        ref_intervals,
        hyp_intervals,
        alignment
    )

    all_start_errors.extend(s_err)
    all_end_errors.extend(e_err)
    all_duration_errors.extend(d_err)

# ------------------------------------------------------------------
# Final Metrics
# ------------------------------------------------------------------

corpus_per = 100 * (total_S + total_D + total_I) / total_N

mean_start = np.mean(all_start_errors) * 1000
median_start = np.median(all_start_errors) * 1000

def tolerance(errors, threshold_ms):
    return np.mean(
        [e <= threshold_ms/1000 for e in errors]
    ) * 100

print(f"\nCorpus PER: {corpus_per:.2f}%")
print(f"Mean boundary error: {mean_start:.2f} ms")
print(f"Median boundary error: {median_start:.2f} ms")
print(f"% within 20ms: {tolerance(all_start_errors, 20):.2f}%")
print(f"% within 50ms: {tolerance(all_start_errors, 50):.2f}%")



Corpus PER: 23.02%
Mean boundary error: 58.41 ms
Median boundary error: 19.88 ms
% within 20ms: 50.36%
% within 50ms: 75.04%


In [125]:
print(len(phones_ref), len(ref_intervals))
print(len(phones_hyp), len(hyp_intervals))

2837 2837
2018 2018


In [126]:
print("REF first start:", ref_intervals[0][0])
print("HYP first start:", hyp_intervals[0][0])


REF first start: 0.3267
HYP first start: 0.0
